# Clasificar imágenes con el modelo ya entrenado

Este notebook **no entrena**: descarga el modelo publicado en el Release más reciente del repo y clasifica imágenes tomadas de enlaces. No hace falta adjuntar nada en Kaggle; solo activar **Internet**. También funciona en Google Colab.

- **Modelo:** por defecto, `modelo.joblib` del último Release de GitHub (ver README, "Publicar el modelo").
- **Entradas:** enlaces a imágenes (GitHub, Google Drive con "cualquier persona con el enlace", etc.) o un `.txt` con un enlace por línea, como `entradas/ejemplos.txt`.

In [ ]:
import os, subprocess, sys

# Funciona en Kaggle y en Google Colab
REPO = "/kaggle/working/repo" if os.path.isdir("/kaggle") else "/content/repo"
if not os.path.isdir(REPO):
    r = subprocess.run(["git", "clone", "https://github.com/11909/Clasificador-fruta-verdura.git", REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("No se pudo clonar el repo. En Kaggle activa Settings → Internet "
                           "(pide verificar tu teléfono) y vuelve a correr esta celda.\n" + r.stderr)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"])
os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image"])  # el resto ya viene en Kaggle/Colab

## 1. Cargar el modelo (una sola vez)

In [ ]:
from src.clasificador import Clasificador

MODELO = None   # None = Release más reciente. También acepta una ruta o enlace a otro modelo.joblib
clf = Clasificador(MODELO)
print(clf.descripcion)

## 2. Elegir las entradas

In [ ]:
# Un .txt con un enlace por línea (este vive en el repo; pueden crear el suyo en cualquier sitio)
ENTRADAS = "https://raw.githubusercontent.com/11909/Clasificador-fruta-verdura/main/entradas/ejemplos.txt"

# O una lista de enlaces sueltos:
# ENTRADAS = [
#     "https://drive.google.com/file/d/<ID>/view?usp=sharing",
#     "https://github.com/<usuario>/<repo>/blob/main/fotos/mango.jpg",
# ]

## 3. Clasificar

In [ ]:
import matplotlib.pyplot as plt
from src.entrada import leer_imagen

resultados = clf.clasificar_lote(ENTRADAS)
buenos = [r for r in resultados if "error" not in r]
for r in resultados:
    if "error" in r:
        print("ERROR", r["entrada"], "→", r["error"])

columnas = 4
filas = max(1, -(-len(buenos) // columnas))
fig, ejes = plt.subplots(filas, columnas, figsize=(3 * columnas, 3.3 * filas), squeeze=False)
for eje in ejes.flat:
    eje.axis("off")
for eje, r in zip(ejes.flat, buenos):
    eje.imshow(leer_imagen(r["entrada"]))
    eje.set_title(f"{r['etiqueta']} ({r['confianza']:.0%})")
plt.tight_layout()

## Opcional: interfaz gráfica

Abre una página con enlace público temporal donde se puede subir una foto o pegar un enlace.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"])
from app.app import crear_demo
crear_demo(clf).launch(share=True)